# Lab 6 — Make something actually happen
### EP0418 · Topic 7 · 2 practical hours

**You need:** `ep0418.py` and your track's data.

---
Everything so far produced numbers. Today a number causes an **action**.

By the end of this session something must leave your notebook — a message, an
email, or a dashboard row that a named person could receive.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

CSV = "smart_building_sensors.csv"     # or your own track's file
fetch(CSV)

df = pd.read_csv(CSV, parse_dates=["timestamp"])
df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan
df = df.drop_duplicates(subset="timestamp", keep="first").sort_values("timestamp")

## Task 1 — Write the sentence first

Fill in all three blanks before you write any code.

> When **____**, the system will **____**, and **____** will receive it.

If you cannot name a specific recipient, fix that before continuing — you do not
yet have a decision system.

**When:**

**The system will:**

**Received by:**

**Cost of a false alarm, and who pays it:**

## Task 2 — Build the rule, with debouncing

A naive rule fires on every single reading that crosses the line. Real systems
require the condition to **hold**.

> Flag periods where `co2_ppm` has been above 1000 for at least 15 consecutive
> minutes (3 consecutive 5-minute readings), rather than flagging single readings.

In [ ]:
# TODO(you): `fires` = a boolean Series, True only when the condition has held
fires = ...

naive_count = (df.co2_ppm > 1000).sum()
if filled(fires):
    print(f"without debouncing: {naive_count:,} readings above the line")
    print(f"with debouncing   : {fires.sum():,} readings during a held condition")
    starts = fires & ~fires.shift(1, fill_value=False)
    print(f"                    {starts.sum():,} separate alert events")

## Task 3 — Rate-limit it

Even debounced, a bad afternoon can send ninety messages. Cap it at **one alert
per hour per condition**.

An *alert event* is one row per alert, with the columns `timestamp`, `co2_ppm`
and (later) `message`.

> `fires` is a boolean Series aligned to `df`. Build `alerts`: a DataFrame with
> one row per alert, columns `timestamp` and `co2_ppm`, keeping at most one
> alert per clock hour.

In [ ]:
# TODO(you): `alerts` = a DataFrame of alert events, at most one per hour
alerts = ...

if filled(alerts):
    print(f"after rate limiting: {len(alerts)} alerts over {(df.timestamp.max()-df.timestamp.min()).days} days")

## Task 4 — Write the message a human will read

Compare:

- `"ALERT"`
- `"CO₂ 1240 ppm in Lab 3, sustained 20 min — check ventilation"`

The second names the value, the place, the duration, and the action. Write yours
to that standard.

In [ ]:
# TODO(you): build `messages` — a list of human-readable alert strings
messages = ...

if filled(messages):
    for m in messages[:5]:
        print(m)

## Task 5 — Actually send one

Pick **one** of the three and make it work. All are free.

**Telegram** (most students choose this):

1. In Telegram, message `@BotFather`, send `/newbot`, follow the prompts, and
   copy the token it gives you (a long string with a colon in it).
2. Send your new bot any message from your own account.
3. In a browser, open `https://api.telegram.org/bot<TOKEN>/getUpdates` with your
   token in place of `<TOKEN>`. Find `"chat":{"id": ...}`; that number is your
   `chat_id`.
4. Send a message from Python:
   `requests.get(f"https://api.telegram.org/bot{TOKEN}/sendMessage", params={"chat_id": CHAT_ID, "text": message})`

**Never leave your token in a notebook you submit.** Put it in Colab's Secrets
(the key icon on the left) or in a cell you delete before submitting.

**Email**: Python's built-in `smtplib`. Good when the recipient is not you.

**Dashboard**: Streamlit, deployed free to a public URL. Good when the response is
"a human should come and look".

In [ ]:
# TODO(you): send at least one real notification.
#            Keep evidence — the API response, or the list of messages sent.
sent = ...

# ---- verification contract — do not edit ----
check.response_fired(sent, minimum=1)
check.report()

---
## Read before you run

An assistant was asked to fire only when CO2 has been above 1000 ppm for three consecutive readings. It wrote:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

Is “the average of three readings is above 1000” the same as “all three readings are above 1000”? Give three numbers that make this rule fire when it should not.

Write down what the rule counts over the full period compared with the correct debounce.

In [ ]:
# --- AI-generated. Read it first. ---
# fires = df.co2_ppm.rolling(3).mean() > 1000

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

- A real notification has arrived somewhere you can show your tutor.
- `verification_log.md`: how many alerts did the naive rule fire, versus your
  debounced and rate-limited version? What would the naive version have cost the
  recipient?
- A **Decisions** line: your threshold, debounce and rate limit, who receives the
  alert, and why.